# NYC Taxi analytics pipeline - ComPar 26/27, Project 1

Ingestion and cleaning (M1), temporal demand (M2), zone statistics (M3)
and the optional operations (O1-O4). Every operation runs inside its own
`with profiler.profile(...)` block, and at the end all metrics (totals and
per-stage breakdown) are saved to `results/<RUN_NAME>.json`.
`analysis.ipynb` reads those files to build the report's tables and charts.

**Two ways to run it:**

- **Interactively:** edit the _Parameters_ cell, then _Kernel -> Restart
  Kernel and Run All Cells_. Restarting matters, because `getOrCreate()`
  reuses an existing Spark session and ignores any new configuration.
- **From the command line** (required by the brief): papermill overrides
  the _Parameters_ cell. The executed copy, with all outputs, is saved to
  `results/notebooks/`.

  ```bash
  scripts/run_pipeline.sh local   -p YEARS 2024 -p FRACTION 0.1 -p RUN_NAME local-smoke
  scripts/run_pipeline.sh cluster -p YEARS 2024 -p RUN_NAME cluster-3w-baseline
  ```


## Parameters

This cell is tagged `parameters`, so papermill can override any of these values with `-p NAME VALUE`.


In [87]:
MASTER = "local[*]"            # "spark://spark-master:7077" inside the Docker cluster
DATA_DIR = None                # None -> ../data locally, /data on the cluster
OUTPUT_DIR = None              # None -> ../output locally, /output on the cluster
RESULTS_DIR = None             # None -> ../results locally, /results on the cluster
YEARS = "2024-2025"                 # "2024" | "2019-2024" | "2024,2026"
MONTHS = "1-12"                # "1-12" | "1-3"
FRACTION = None                # sample fraction e.g. 0.1, or None for every record
OPS = "m2,m3,o2,o3"            # operations to run after M1 (M1 always runs)
RUN_NAME = None                # label for results/<RUN_NAME>.json; None -> timestamp
EXPLAIN = False                # save the formatted physical plan of every operation
DRIVER_MEMORY = "8g"           # local mode: the driver is also the only executor
EXECUTOR_MEMORY = "3g"         # cluster mode: must fit within the workers' --memory
SPARK_CONF = {}                # non-default settings, e.g. {"spark.sql.shuffle.partitions": "64"}

# O4 classification thresholds (fraction of a bucket's trips)
AIRPORT_THRESHOLD = 0.3
NEGOTIATED_THRESHOLD = 0.1
GROUP_THRESHOLD = 0.2

In [88]:
import os
import time

on_cluster = MASTER.startswith("spark://")
DATA_DIR = DATA_DIR or ("/data" if on_cluster else "../data")
OUTPUT_DIR = OUTPUT_DIR or ("/output" if on_cluster else "../output")
RESULTS_DIR = RESULTS_DIR or ("/results" if on_cluster else "../results")
RUN_NAME = RUN_NAME or time.strftime("run-%Y%m%d-%H%M%S")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


def parse_range(spec, lo, hi):
    """'2024' -> [2024]; '2019-2021' -> [2019, 2020, 2021]; '2019,2023' -> [2019, 2023]."""
    values = []
    for part in str(spec).split(","):
        if "-" in part:
            a, b = part.split("-")
            values.extend(range(int(a), int(b) + 1))
        else:
            values.append(int(part))
    return sorted(v for v in set(values) if lo <= v <= hi)


years = parse_range(YEARS, 2009, 2026)
months = parse_range(MONTHS, 1, 12)
ops = [o.strip().lower() for o in str(OPS).split(",") if o.strip()]
print(f"run={RUN_NAME} years={years} months={months} fraction={FRACTION} ops={ops}")
print(f"data={DATA_DIR} output={OUTPUT_DIR} results={RESULTS_DIR}")

run=run-20261015-120952 years=[2024, 2025] months=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] fraction=None ops=['m2', 'm3', 'o2', 'o3']
data=../data output=../output results=../results


## Spark session

The baseline deliberately uses Spark defaults. Each experiment's
non-default settings go in `SPARK_CONF`, so they are recorded in the
results file.

**Java:** Spark 4.0 runs only on Java 17 or 21. Java 23+ fails with
`UnsupportedOperationException: getSubject is not supported`. Jupyter and
PyCharm kernels do not inherit a `JAVA_HOME` exported in your terminal, so
the next cell picks a supported JDK itself (local mode only). If the
session already failed once in this kernel, **restart the kernel**: the
JVM has already started with the wrong Java.


In [89]:
import platform
import subprocess

SUPPORTED_JAVA = (17, 21)


def java_major(java_home):
    """Major version of the JDK at `java_home`, read from its `release` file."""
    try:
        with open(os.path.join(java_home, "release")) as f:
            for line in f:
                if line.startswith("JAVA_VERSION="):
                    return int(line.split("=", 1)[1].strip().strip('"').split(".")[0])
    except (OSError, ValueError):
        pass
    return None


if not on_cluster and java_major(os.environ.get("JAVA_HOME", "")) not in SUPPORTED_JAVA:
    if platform.system() == "Darwin":
        for version in ("21", "17"):
            found = subprocess.run(["/usr/libexec/java_home", "-v", version],
                                   capture_output=True, text=True)
            if found.returncode == 0:
                os.environ["JAVA_HOME"] = found.stdout.strip()
                break
    if java_major(os.environ.get("JAVA_HOME", "")) not in SUPPORTED_JAVA:
        raise RuntimeError("Spark 4.0 needs Java 17 or 21: install one and set JAVA_HOME "
                           "before starting Jupyter")

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "(cluster: Java from the Docker image)"))

JAVA_HOME: /Users/rloucao/Library/Java/JavaVirtualMachines/ms-21.0.8/Contents/Home


In [90]:
from pyspark.sql import SparkSession

builder = (SparkSession.builder
           .master(MASTER)
           .appName(f"taxi-{RUN_NAME}")
           .config("spark.driver.memory", DRIVER_MEMORY)
           .config("spark.ui.showConsoleProgress", "false")).config("spark.sql.session.timeZone", "UTC")


if on_cluster:
    builder = builder.config("spark.executor.memory", EXECUTOR_MEMORY)
for key, value in SPARK_CONF.items():
    builder = builder.config(key, value)

spark = builder.getOrCreate()
sc = spark.sparkContext

print("Spark version:", spark.version)
print("Spark UI:", sc.uiWebUrl)
print("Default parallelism (cores seen by the driver):", sc.defaultParallelism)

Spark version: 4.0.1
Spark UI: http://172.20.10.11:4040
Default parallelism (cores seen by the driver): 10


In [91]:
import sys
sys.path.append(os.path.abspath("."))   # spark_profiler.py lives next to this notebook

from spark_profiler import SparkProfiler

profiler = SparkProfiler(spark)

## Ingestion

The monthly files are **not schema-compatible** across 2009-2026:

- column names changed (2009 uses `Trip_Pickup_DateTime`, `Fare_Amt`, ...;
  2010 uses `pickup_datetime`, ...);
- types changed too (`passenger_count` is a double in some years and
  int64 in others).

Reading everything with a single `spark.read.parquet(glob)` either fails
or silently produces nulls. Instead, each file is read on its own, mapped
onto the canonical schema below, and the results are unioned.

**Note:** the 2009-2010 files have pickup/dropoff lat/lon instead of
`PULocationID`/`DOLocationID`. Those columns come out null for these
years, so M1 removes their rows. Decide (and document in the report)
whether that is acceptable, or restrict the study to 2011+.


In [92]:
from functools import reduce

import pyspark.sql.functions as F
from pyspark.sql import DataFrame, Window
from pyspark.sql.types import (DoubleType, IntegerType, StringType, StructField,
                               StructType, TimestampType)

# Canonical (name, type) of every column the pipeline uses.
CANONICAL_COLUMNS = [
    ("tpep_pickup_datetime", TimestampType()),
    ("tpep_dropoff_datetime", TimestampType()),
    ("PULocationID", IntegerType()),
    ("DOLocationID", IntegerType()),
    ("passenger_count", IntegerType()),
    ("trip_distance", DoubleType()),
    ("RatecodeID", IntegerType()),
    ("payment_type", IntegerType()),
    ("fare_amount", DoubleType()),
    ("tip_amount", DoubleType()),
    ("total_amount", DoubleType()),
]

# Older spellings of the canonical columns (matched case-insensitively).
# TODO: verify against the 2009/2010 files; their payment_type is a string
# ("CASH", "Credit", ...) and needs mapping to the numeric codes.
COLUMN_ALIASES = {
    "tpep_pickup_datetime": ["Trip_Pickup_DateTime", "pickup_datetime"],
    "tpep_dropoff_datetime": ["Trip_Dropoff_DateTime", "dropoff_datetime"],
    "RatecodeID": ["Rate_Code", "rate_code"],
    "fare_amount": ["Fare_Amt"],
    "tip_amount": ["Tip_Amt"],
    "total_amount": ["Total_Amt"],
}

ZONES_SCHEMA = StructType([
    StructField("LocationID", IntegerType(), False),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])

In [93]:
def month_files(data_dir, years, months):
    """(year, month, path) for every requested month that exists on disk."""
    found = []
    for y in years:
        for m in months:
            path = os.path.join(data_dir, f"yellow_tripdata_{y}-{m:02d}.parquet")
            if os.path.exists(path):
                found.append((y, m, path))
            else:
                print(f"[ingest] missing {path} - skipped")
    if not found:
        raise FileNotFoundError(f"no yellow_tripdata files for years={years} in {data_dir}")
    return found


def read_month(path, year, month):
    df = spark.read.parquet(path)
    by_lower = {c.lower(): c for c in df.columns}
    exprs = []
    for name, dtype in CANONICAL_COLUMNS:
        candidates = [name] + COLUMN_ALIASES.get(name, [])
        src = next((by_lower[c.lower()] for c in candidates if c.lower() in by_lower), None)
        col = F.col(src) if src is not None else F.lit(None)
        exprs.append(col.cast(dtype).alias(name))
    # The file's own year/month: pickup timestamps contain stray out-of-range
    # dates, so per-year statistics are keyed on the source file instead.
    return df.select(*exprs, F.lit(year).alias("source_year"), F.lit(month).alias("source_month"))


def load_trips(data_dir, years, months, fraction=None, seed=42):
    files = month_files(data_dir, years, months)
    print(f"[ingest] reading {len(files)} monthly files")
    trips = reduce(DataFrame.unionByName, (read_month(path, y, m) for y, m, path in files))
    if fraction is not None and fraction < 1.0:
        trips = trips.sample(fraction=fraction, seed=seed)
    return trips


def load_zones(path):
    return spark.read.option("header", "true").schema(ZONES_SCHEMA).csv(path)

In [94]:
zones = load_zones(os.path.join(DATA_DIR, "taxi_zone_lookup.csv"))
trips = load_trips(DATA_DIR, years, months, FRACTION)

print("trips partitions:", trips.rdd.getNumPartitions())
trips.printSchema()

[ingest] reading 24 monthly files
trips partitions: 240
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- source_year: integer (nullable = false)
 |-- source_month: integer (nullable = false)



## M1 - Data ingestion and cleaning

A record is removed if it violates **any** of these criteria. They are not
mutually exclusive, so the per-criterion counts can add up to more than
`removed`, which is reported separately.


In [95]:
LOCATION_MIN, LOCATION_MAX = 1, 263

# Removed the "passanger_count" column because it is not required
# Only O1 uses it, so instead of filtering it out, we can just ignore it in the criteria
# About 10% of the records have passenger_count=0, which is invalid, but we don't need to filter them out for m2/m3
REQUIRED = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID",
            "DOLocationID", "trip_distance", "fare_amount"]


def criteria():
    """name -> boolean Column that is True when the record is invalid."""
    pu, do, pc = F.col("PULocationID"), F.col("DOLocationID"), F.col("passenger_count")
    raw = {
        "null_required": reduce(lambda a, b: a | b, (F.col(c).isNull() for c in REQUIRED)),
        "fare_le_0": F.col("fare_amount") <= 0,
        "distance_le_0": F.col("trip_distance") <= 0,
        "bad_passenger_count": (pc == 0) | (pc > 6),
        "dropoff_not_after_pickup": F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"),
        "location_out_of_range": ~pu.between(LOCATION_MIN, LOCATION_MAX)
                                 | ~do.between(LOCATION_MIN, LOCATION_MAX),
                                         "pickup_outside_file_month": (F.year("tpep_pickup_datetime") != F.col("source_year"))
                                     | (F.month("tpep_pickup_datetime") != F.col("source_month")),
                                 
                                
    }
    # Comparisons against null yield null; nulls are covered by `null_required`.
    return {name: F.coalesce(cond, F.lit(False)) for name, cond in raw.items()}


def is_invalid():
    return reduce(lambda a, b: a | b, criteria().values())


def cleaning_stats(trips):
    """Per source year: total records, removed per criterion, total removed."""
    aggs = [F.count(F.lit(1)).alias("total")]
    aggs += [F.sum(F.when(cond, 1).otherwise(0)).alias(name) for name, cond in criteria().items()]
    aggs += [F.sum(F.when(is_invalid(), 1).otherwise(0)).alias("removed")]
    return trips.groupBy("source_year").agg(*aggs).orderBy("source_year")


def clean_trips(trips):
    return trips.filter(~is_invalid())

In [96]:
import pandas as pd

with profiler.profile("m1"):
    m1_rows = cleaning_stats(trips).collect()   # tiny: one row per year

m1_stats = pd.DataFrame([r.asDict() for r in m1_rows]).set_index("source_year")
m1_stats.to_csv(os.path.join(OUTPUT_DIR, "m1_cleaning_stats.csv"))

# Lazy: every operation below re-reads and re-filters the Parquet files.
# That is the naive baseline on purpose (see caching in the optimisation study).
clean = clean_trips(trips)
m1_stats

[m1] wall time: 5.73s (4 stages, 243 tasks) - see profiler.report() / profiler.compare(...)


,total,null_required,fare_le_0,distance_le_0,bad_passenger_count,dropoff_not_after_pickup,location_out_of_range,pickup_outside_file_month,removed
source_year,,,,,,,,,
2024,41169720,0,748284,776305,401638,13510,425660,420,2210549
2025,48722602,0,2870234,1402958,260208,546304,371657,214,5122979


## Shared helpers

`run_operation` builds an operation's (lazy) result outside the profiler,
then profiles only the write, which is the action that actually runs the
jobs. Operations that are not in `OPS`, or not implemented yet, are skipped.


In [97]:
import contextlib
import io


def with_time_features(df):
    """Derived columns used by several operations."""
    pickup, dropoff = F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")
    return (df
            .withColumn("pickup_date", F.to_date(pickup))
            .withColumn("day_of_week", F.dayofweek(pickup))          # 1 = Sunday ... 7 = Saturday
            .withColumn("hour_of_day", F.hour(pickup))
            .withColumn("pickup_month", F.date_trunc("month", pickup))
            .withColumn("duration_min",
                        (F.unix_timestamp(dropoff) - F.unix_timestamp(pickup)) / 60.0))


def with_pickup_zone(df, zones):
    """Annotate trips with pickup Borough/Zone/service_zone (zone join #1)."""
    pu = zones.select(F.col("LocationID").alias("PULocationID"),
                      F.col("Borough").alias("pickup_borough"),
                      F.col("Zone").alias("pickup_zone"),
                      F.col("service_zone").alias("pickup_service_zone"))
    return df.join(pu, on="PULocationID", how="left")


def with_dropoff_zone(df, zones):
    """Annotate trips with dropoff Borough/Zone/service_zone (zone join #2)."""
    do = zones.select(F.col("LocationID").alias("DOLocationID"),
                      F.col("Borough").alias("dropoff_borough"),
                      F.col("Zone").alias("dropoff_zone"),
                      F.col("service_zone").alias("dropoff_service_zone"))
    return df.join(do, on="DOLocationID", how="left")


def run_operation(name, build, write):
    """Build `name`'s lazy result, then profile its write (the action)."""
    if name not in ops:
        print(f"[{name}] not in OPS - skipped")
        return None
    try:
        result = build()
    except NotImplementedError as e:
        print(f"[{name}] skipped: {e}")
        return None
    with profiler.profile(name):
        write(result)
    if EXPLAIN:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            result.explain(mode="formatted")
        with open(os.path.join(RESULTS_DIR, f"{RUN_NAME}-{name}-plan.txt"), "w") as f:
            f.write(buf.getvalue())
    return result

## M2 - Temporal demand analysis

For each `(pickup_borough, day_of_week, hour_of_day)` triple compute:

- total trip count
- mean `fare_amount`
- mean trip duration in minutes (derived from the timestamps)
- mean `trip_distance`

The output is written as Parquet, **partitioned by borough**.

_Performance angle:_ zone join (broadcast vs. sort-merge), group-by shuffle, partitioned write (number of output files).


In [98]:
# (no actions here - only transformations)
def build_m2(clean: DataFrame, zones: DataFrame):
    trips = with_pickup_zone(with_time_features(clean), zones)
    return (trips
            .groupBy("pickup_borough", "day_of_week", "hour_of_day")
            .agg(F.count("*").alias("trip_count"),
                 F.avg("fare_amount").alias("avg_fare"),
                 F.avg("duration_min").alias("avg_duration_min"),
                 F.avg("trip_distance").alias("avg_distance")))
    

In [99]:
m2_result = run_operation("m2", lambda: build_m2(clean, zones),
                        write=lambda df: df.write.mode("overwrite").partitionBy("pickup_borough").parquet(f"{OUTPUT_DIR}/m2"))

26/10/15 12:10:14 WARN DAGScheduler: Broadcasting large task binary with size 1004.1 KiB


[m2] wall time: 13.64s (3 stages, 245 tasks) - see profiler.report() / profiler.compare(...)


## M3 - Zone-level statistics

For each pickup zone and month: mean and standard deviation of
`fare_amount` and `trip_distance`. Then, using window functions:

- rank zones by monthly trip volume within each borough
- compute each zone's rolling 3-month trip share within its borough
- flag zones whose current-month volume deviates from their own rolling
  mean by more than 1.5 standard deviations

_Performance angle:_ several windows with different partitionings, each adding an Exchange + Sort; window partition sizes.


In [100]:
def build_m3(clean: DataFrame, zones: DataFrame):
    # Part A: one row per (zone, month); "month" is the source file's month
    monthly = (with_pickup_zone(clean, zones)
               .withColumn("month", F.make_date("source_year", "source_month", F.lit(1)))
               .withColumn("month_idx", F.col("source_year") * 12 + F.col("source_month"))
               .groupBy("pickup_borough", "PULocationID", "pickup_zone", "month", "month_idx")
               .agg(F.count("*").alias("trip_count"),
                    F.avg("fare_amount").alias("avg_fare"),
                    F.stddev("fare_amount").alias("std_fare"),
                    F.avg("trip_distance").alias("avg_distance"),
                    F.stddev("trip_distance").alias("std_distance")))

    # Part B: windows (rangeBetween works on month numbers, so missing months are handled)
    by_borough_month = Window.partitionBy("pickup_borough", "month").orderBy(F.desc("trip_count"))
    zone_last_3m = Window.partitionBy("PULocationID").orderBy("month_idx").rangeBetween(-2, 0)
    borough_last_3m = Window.partitionBy("pickup_borough").orderBy("month_idx").rangeBetween(-2, 0)
    zone_prev_3m = Window.partitionBy("PULocationID").orderBy("month_idx").rangeBetween(-3, -1)

    return (monthly
            .withColumn("volume_rank", F.rank().over(by_borough_month))
            .withColumn("rolling_3m_share",
                        F.sum("trip_count").over(zone_last_3m) / F.sum("trip_count").over(borough_last_3m))
            .withColumn("prev_3m_mean", F.avg("trip_count").over(zone_prev_3m))
            .withColumn("prev_3m_std", F.stddev("trip_count").over(zone_prev_3m))
            .withColumn("prev_3m_months", F.count("trip_count").over(zone_prev_3m))
            .withColumn("volume_anomaly",
                        (F.col("prev_3m_months") == 3)
                        & (F.abs(F.col("trip_count") - F.col("prev_3m_mean")) > 1.5 * F.col("prev_3m_std"))))

In [101]:
m3_result = run_operation("m3", lambda: build_m3(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/m3"))

[m3] wall time: 9.93s (7 stages, 248 tasks) - see profiler.report() / profiler.compare(...)


## O1 - Fare anomaly detection

For each `(PULocationID, hour_of_day)` bucket compute the mean and σ of
`fare_amount / trip_distance`. Flag trips whose per-mile fare exceeds
mean + 3σ, then compare weekday vs. weekend anomaly rates by
`payment_type` and pickup borough.

_Performance angle:_ aggregate, then join the bucket statistics back to the full trip table.


In [102]:
def build_o1(clean: DataFrame, zones: DataFrame):
    # TODO: implement - return a lazy DataFrame (no actions here)
    raise NotImplementedError("o1 not implemented yet")

In [103]:
o1_result = run_operation("o1", lambda: build_o1(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/o1"))

[o1] not in OPS - skipped


## O2 - Origin-destination flow analysis

- Monthly trip counts for all `(PULocationID, DOLocationID)` pairs, with
  the zone lookup joined to resolve zone names and boroughs.
- Each route's share of total outbound trips from its pickup zone per month.
- Top-20 routes per pickup borough, and routes whose monthly share
  reverses direction between Q1 and Q3.

_Performance angle:_ large shuffle on a high-cardinality key, two zone joins, skew (Manhattan and airport routes).


In [104]:
ROUTE = ["PULocationID", "DOLocationID"]

def build_o2(clean: DataFrame, zones: DataFrame):
    # 1. monthly trips per route, then both ends' zone names (join after aggregating)
    routes = (clean
              .groupBy("source_year", "source_month", *ROUTE)
              .agg(F.count("*").alias("trip_count")))
    routes = with_dropoff_zone(with_pickup_zone(routes, zones), zones)

    # 2. share of the pickup zone's outbound trips that month
    outbound = Window.partitionBy("source_year", "source_month", "PULocationID")
    routes = routes.withColumn("outbound_share",
                               F.col("trip_count") / F.sum("trip_count").over(outbound))

    # 3. top-20 routes per pickup borough, by total trips over the whole period
    by_borough = Window.partitionBy("pickup_borough").orderBy(F.desc("route_trips"))
    top20 = (routes.groupBy("pickup_borough", *ROUTE)
             .agg(F.sum("trip_count").alias("route_trips"))
             .withColumn("borough_rank", F.row_number().over(by_borough))
             .filter(F.col("borough_rank") <= 20))

    # 4. direction of the share within Q1 (Jan -> Mar) and Q3 (Jul -> Sep)
    trend = (routes.groupBy("source_year", *ROUTE)
             .pivot("source_month", [1, 3, 7, 9])
             .agg(F.first("outbound_share"))
             .na.fill(0.0)
             .withColumn("q1_change", F.col("3") - F.col("1"))
             .withColumn("q3_change", F.col("9") - F.col("7"))
             .withColumn("share_reverses", F.col("q1_change") * F.col("q3_change") < 0)
             .select("source_year", *ROUTE, "q1_change", "q3_change", "share_reverses"))

    # 5. one table: every route-month, with its top-20 rank and Q1/Q3 trend
    return (routes
            .join(top20.select(*ROUTE, "route_trips", "borough_rank"), on=ROUTE, how="left")
            .join(trend, on=["source_year", *ROUTE], how="left")
            .withColumn("is_top20", F.col("borough_rank").isNotNull()))

In [105]:
o2_result = run_operation("o2", lambda: build_o2(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/o2"))

[o2] wall time: 12.71s (14 stages, 321 tasks) - see profiler.report() / profiler.compare(...)


In [106]:
o2 = pd.read_parquet("../output/o2")
top = o2[o2.is_top20].drop_duplicates(ROUTE)
top[top.pickup_borough == "Manhattan"].sort_values("borough_rank").head()

,source_year,PULocationID,DOLocationID,source_month,trip_count,pickup_borough,pickup_zone,pickup_service_zone,dropoff_borough,dropoff_zone,dropoff_service_zone,outbound_share,route_trips,borough_rank,q1_change,q3_change,share_reverses,is_top20
5789,2024,237,236,7,16022,Manhattan,Upper East Side South,Yellow Zone,Manhattan,Upper East Side North,Yellow Zone,0.129407,560709.0,1.0,-0.015349,0.023168,True,True
18653,2025,236,237,2,19417,Manhattan,Upper East Side North,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone,0.139372,480239.0,2.0,-0.007018,0.023148,True,True
59966,2024,237,237,7,11410,Manhattan,Upper East Side South,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone,0.092157,376848.0,3.0,-0.006936,0.009021,True,True
44527,2025,236,236,2,15382,Manhattan,Upper East Side North,Yellow Zone,Manhattan,Upper East Side North,Yellow Zone,0.110409,357942.0,4.0,-0.009718,0.010049,True,True
147084,2025,161,237,9,10763,Manhattan,Midtown Center,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone,0.069990,264129.0,5.0,-0.005176,0.005003,True,True


## O3 - Demand forecasting features

For each `(PULocationID, day_of_week, hour_of_day)` bucket: mean historical
trip count, variance, and a 4-week-lag value modelled as a **self-join on
offset pickup dates**. Demand index = current count ÷ historical mean;
flag buckets with an index above 2.0. Join the zone lookup to annotate
results by `service_zone`.

_Performance angle:_ self-join (shuffles both sides), window partition sizing.


In [107]:
BUCKET = ["PULocationID", "day_of_week", "hour_of_day"]


def build_o3(clean: DataFrame, zones: DataFrame):
    # 1. trips per (zone, date, hour); hours with no trips have no row
    hourly = (with_time_features(clean)
              .groupBy("PULocationID", "pickup_date", "day_of_week", "hour_of_day")
              .agg(F.count("*").alias("trip_count")))

    # 2. historical mean/variance per bucket, counting zero-trip hours:
    #    divide by how many of that weekday the data covers, not by the rows found
    n_days = (hourly.select("pickup_date", "day_of_week").distinct()
              .groupBy("day_of_week").agg(F.count("*").alias("n_days")))
    stats = (hourly.groupBy(*BUCKET)
             .agg(F.sum("trip_count").alias("sum_trips"),
                  F.sum(F.col("trip_count") * F.col("trip_count")).alias("sum_sq"))
             .join(n_days, on="day_of_week")
             .withColumn("hist_mean", F.col("sum_trips") / F.col("n_days"))
             .withColumn("hist_var", (F.col("sum_sq") - F.col("n_days") * F.col("hist_mean") ** 2)
                                     / (F.col("n_days") - 1))
             .select(*BUCKET, "hist_mean", "hist_var"))

    # 3. 4-week lag as a self-join: shift a copy 28 days forward, join on equal dates
    lag = hourly.select("PULocationID", "hour_of_day",
                        F.date_add("pickup_date", 28).alias("pickup_date"),
                        F.col("trip_count").alias("lag_4w_count"))
    first_date = hourly.agg(F.min("pickup_date").alias("first_date"))   # one row

    result = (hourly
              .join(lag, on=["PULocationID", "pickup_date", "hour_of_day"], how="left")
              # no match = 0 trips, unless 4 weeks earlier is before the data starts (null)
              .crossJoin(first_date)
              .withColumn("lag_4w_count",
                          F.when(F.date_sub("pickup_date", 28) >= F.col("first_date"),
                                 F.coalesce("lag_4w_count", F.lit(0))))
              .drop("first_date")
              # 4. demand index and flag
              .join(stats, on=BUCKET, how="left")
              .withColumn("demand_index", F.col("trip_count") / F.col("hist_mean"))
              .withColumn("high_demand", F.col("demand_index") > 2.0))

    # 5. annotate with service_zone (plus borough and zone name)
    return with_pickup_zone(result, zones)

In [108]:
o3_result = run_operation("o3", lambda: build_o3(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/o3"))

26/10/15 12:10:39 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/10/15 12:12:26 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/10/15 12:12:27 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
26/10/15 12:12:31 WARN DAGScheduler: Broadcasting large task binary with size 6.5 MiB


[o3] wall time: 115.82s (14 stages, 1314 tasks) - see profiler.report() / profiler.compare(...)


## O4 - Trip-type classification

Per `(PULocationID, hour_of_day)` bucket: fraction of airport trips
(`RatecodeID` ∈ {2, 3}), negotiated fares (`RatecodeID` = 5) and group
trips (`passenger_count` ≥ 4). Classify each bucket as residential /
commercial / transit / mixed using `AIRPORT_THRESHOLD`,
`NEGOTIATED_THRESHOLD` and `GROUP_THRESHOLD`. Join the classification back
to the full trip table and compute fare and distance statistics per
zone type per month.

_Performance angle:_ join-back of a small table to a huge one, Python UDF vs. built-in `when` chain.


In [109]:
def build_o4(clean: DataFrame, zones: DataFrame):
    # TODO: implement - return a lazy DataFrame (no actions here)
    raise NotImplementedError("o4 not implemented yet")

In [110]:
o4_result = run_operation("o4", lambda: build_o4(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/o4"))

[o4] not in OPS - skipped


## Summary

Totals per operation for this run. `profiler.stages("<op>")` shows the
per-stage breakdown; `skew_max_over_median` ≥ 10 marks a skewed stage
(one task taking 10× the median).


In [111]:
profiler.report()

,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
m1,5.73,4,4,243,46824,29177,1.1GB,0.0B,19.5KB,9.8KB,0.0B,0.0B
m2,13.64,3,3,245,124788,73335,1.1GB,77.7KB,4.8MB,4.8MB,0.0B,0.0B
m3,9.93,7,7,248,84761,60100,1.1GB,351.3KB,4.8MB,4.8MB,0.0B,0.0B
o2,12.71,14,14,321,114110,73108,1.1GB,15.7MB,204.8MB,150.4MB,0.0B,0.0B
o3,115.82,14,14,1314,1091794,809464,5.5GB,15.7MB,280.9MB,280.9MB,0.0B,0.0B


In [112]:
for name in profiler.runs:
    stages = profiler.stages(name)
    skewed = stages[stages["skew_max_over_median"].fillna(0) >= 10]
    print(f"{name}: {len(stages)} stages, {len(skewed)} skewed")
    if len(skewed):
        display(skewed)

m1: 4 stages, 1 skewed


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
stage_id,,,,,,,,,,,
24,collect at /var/folders/c9/lj6q8jdd421300db1hf...,5008,240,16.0,1005.0,62.81,1189189378,0,9860,0,0


m2: 3 stages, 1 skewed


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
stage_id,,,,,,,,,,,
33,$anonfun$withThreadLocalCaptured$2 at Completa...,13019,240,16.0,2255.0,140.94,1189189378,0,5018741,0,0


m3: 7 stages, 1 skewed


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
stage_id,,,,,,,,,,,
37,$anonfun$withThreadLocalCaptured$2 at Completa...,8924,240,14.0,2285.0,163.21,1189189378,0,3537766,0,0


o2: 14 stages, 1 skewed


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
stage_id,,,,,,,,,,,
61,$anonfun$withThreadLocalCaptured$2 at Completa...,8688,240,15.0,1357.0,90.47,1189189378,0,28516815,0,0


o3: 14 stages, 5 skewed


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
stage_id,,,,,,,,,,,
87,$anonfun$withThreadLocalCaptured$2 at Completa...,8321,240,14.0,1533.0,109.50,1189189378,0,13874,0,0
88,$anonfun$withThreadLocalCaptured$2 at Completa...,19839,240,21.0,1964.0,93.52,1189189378,0,55244461,0,0
89,$anonfun$withThreadLocalCaptured$2 at Completa...,30996,240,12.0,1732.0,144.33,1189189378,0,55244461,0,0
90,$anonfun$withThreadLocalCaptured$2 at Completa...,38208,240,11.0,1049.0,95.36,1189189378,0,93446,0,0
91,$anonfun$withThreadLocalCaptured$2 at Completa...,106255,240,19.0,11221.0,590.58,1189189378,0,52615433,0,0


## Save results

Writes `results/<RUN_NAME>.json`, which holds every operation's totals and
per-stage metrics plus the configuration needed to reproduce the run.


In [113]:
import platform

executors = sc._jsc.sc().statusTracker().getExecutorInfos()
metadata = {
    "run_name": RUN_NAME,
    "master": sc.master,
    "spark_version": spark.version,
    "num_executors": max(len(executors) - 1, 1),   # minus the driver; local mode = 1
    "total_cores": sc.defaultParallelism,
    "years": years, "months": months, "fraction": FRACTION, "ops": ops,
    "data_dir": DATA_DIR, "host": platform.node(),
    "spark_conf": {k: v for k, v in sc.getConf().getAll()
                   if k.startswith(("spark.sql", "spark.executor", "spark.driver.memory",
                                    "spark.default"))
                   and not any(x in k for x in ("extraJavaOptions", "warehouse", "artifact", "executor.id"))},
    "total_wall_time_s": round(sum(r["wall_time_s"] for r in profiler.runs.values()), 2),
}
results_path = os.path.join(RESULTS_DIR, f"{RUN_NAME}.json")
profiler.save(results_path, metadata)
print("saved", results_path, "| total wall time:", metadata["total_wall_time_s"], "s")

saved ../results/run-20261015-120952.json | total wall time: 157.83 s


In [114]:
# Keep the session open while exploring interactively; stop it to release
# the cluster's executors (and always when running through papermill).
spark.stop()